In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path("..").resolve()))
import json
data_dir = Path("../data")

In [ ]:
def process_bambi_annotations(
    ann_file: Path | str,
    output_dir: Path | str,
    flight_id: str,                      
    visibility_threshold: float = 0.3,
    skip_propagated: bool = True,
    img_width: int = 1024,
    img_height: int = 1024
) -> tuple[int, int, dict]:
    """
    Parses a BAMBI MOT annotation file, extracts tracking metadata, 
    and writes YOLO-formatted label files in a single pass.
    
    Returns:
        tuple: (written_frames, empty_frames, track_spans_dictionary)
    """
    ann_file = Path(ann_file)
    output_dir = Path(output_dir)

    SPECIES_MAP = {
        "Sus scrofa (Wild boar)": 0,
        "Cervus elaphus (Red deer)": 1,
        "Capreolus capreolus (Roe deer)": 2,
    }

    yolo_labels: dict[int, list[str]] = {}
    all_frame_ids: set[int] = set()
    track_spans: dict[int, dict] = {}

    with open(ann_file, encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split(",")
            if len(parts) < 13:
                continue

            frame_id      = int(parts[0])
            track_id      = int(parts[1])
            bb_left       = float(parts[2])
            bb_top        = float(parts[3])
            bb_width      = float(parts[4])
            bb_height     = float(parts[5])
            visibility    = float(parts[8])
            species       = parts[9].strip()
            is_propagated = int(parts[12])

            all_frame_ids.add(frame_id)

            if track_id not in track_spans:
                track_spans[track_id] = {
                    "first": frame_id,
                    "last": frame_id,
                    "species": species
                }
            else:
                span = track_spans[track_id]
                span["first"] = min(span["first"], frame_id)
                span["last"]  = max(span["last"], frame_id)

            if skip_propagated and is_propagated == 1:
                continue
            if visibility < visibility_threshold:
                continue
            if species not in SPECIES_MAP:
                continue

            center_x = (bb_left + bb_width  / 2) / img_width
            center_y = (bb_top  + bb_height / 2) / img_height
            norm_w   = bb_width  / img_width
            norm_h   = bb_height / img_height
            class_id = SPECIES_MAP[species]
            label = f"{class_id} {center_x:.6f} {center_y:.6f} {norm_w:.6f} {norm_h:.6f}"
            yolo_labels.setdefault(frame_id, []).append(label)

    output_dir.mkdir(parents=True, exist_ok=True)
    written_count = 0
    for frame_id, labels in yolo_labels.items():
        (output_dir / f"{flight_id}_{frame_id:08d}.txt").write_text("\n".join(labels))
        written_count += 1

    empty_count = len(all_frame_ids) - written_count
    return written_count, empty_count, track_spans

In [ ]:
annotation_files = list(data_dir.glob("*_gt.txt"))
total_written = 0
total_empty = 0
all_tracking_data = {} 

for ann_file in annotation_files:
    flight_id = ann_file.stem.split("_")[0]
    output_dir = data_dir / "labels" / flight_id

    written, empty, track_spans = process_bambi_annotations(ann_file, output_dir, flight_id=flight_id)
    
    total_written += written
    total_empty += empty
    
    all_tracking_data[flight_id] = track_spans
    
    print(f"Flight {flight_id}: {written} YOLO files written, {empty} empty frames skipped.")

print("-" * 40)
print(f"Total YOLO Files Created: {total_written}")
print(f"Total Empty Frames Skipped: {total_empty}")

In [ ]:
import json
from pathlib import Path

def convert_bambi_to_coco(
    data_dir: Path | str, 
    output_file: Path | str,
    visibility_threshold: float = 0.3,
    skip_propagated: bool = True,
    img_width: int = 1024,
    img_height: int = 1024
) -> tuple[int, int, int, dict]:
    """
    Converts BAMBI MOT files to a single COCO JSON, while preserving 
    tracking IDs and returning tracking lifespan metadata.
    
    Returns:
        tuple: (total_images, total_annotations, skipped_annotations, track_spans)
    """
    data_dir = Path(data_dir)
    output_file = Path(output_file)

    SPECIES_MAP = {
        "Sus scrofa (Wild boar)": 0,
        "Cervus elaphus (Red deer)": 1,
        "Capreolus capreolus (Roe deer)": 2,
    }

    coco_data = {
        "images": [],
        "annotations": [],
        "categories": [
            {"id": class_id, "name": name, "supercategory": "animal"} 
            for name, class_id in SPECIES_MAP.items()
        ]
    }

    global_img_id = 1
    global_ann_id = 1
    img_id_map = {}
    skipped_annotations = 0

    track_spans: dict[tuple[str, int], dict] = {}

    for ann_file in sorted(data_dir.glob("*_gt.txt")):  
        flight_id = ann_file.stem.split('_')[0]         
                                                         
        with open(ann_file, encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split(",")
                if len(parts) < 13:
                    continue

                frame_idx     = int(parts[0])
                track_id      = int(parts[1])
                bb_left       = float(parts[2])
                bb_top        = float(parts[3])
                bb_width      = float(parts[4])
                bb_height     = float(parts[5])
                visibility    = float(parts[8])
                species       = parts[9].strip()
                is_propagated = int(parts[12])

                span_key = (flight_id, track_id)
                if span_key not in track_spans:
                    track_spans[span_key] = {
                        "first": frame_idx,
                        "last": frame_idx,
                        "species": species,
                        "flight_id": flight_id
                    }
                else:
                    span = track_spans[span_key]
                    span["first"] = min(span["first"], frame_idx)
                    span["last"]  = max(span["last"], frame_idx)

                if skip_propagated and is_propagated == 1:
                    skipped_annotations += 1
                    continue
                if visibility < visibility_threshold:
                    skipped_annotations += 1
                    continue
                if species not in SPECIES_MAP:
                    skipped_annotations += 1
                    continue

                img_key = f"{flight_id}_{frame_idx}"
                if img_key not in img_id_map:
                    img_id_map[img_key] = global_img_id
                    file_name = f"frames_detection/{flight_id}/thermal/{flight_id}_{frame_idx:08d}.png"
                    coco_data["images"].append({
                        "id": global_img_id,
                        "file_name": file_name,
                        "width": img_width,
                        "height": img_height
                    })
                    global_img_id += 1

                current_image_id = img_id_map[img_key]
                area = bb_width * bb_height

                coco_data["annotations"].append({
                    "id": global_ann_id,
                    "image_id": current_image_id,
                    "category_id": SPECIES_MAP[species],
                    "track_id": track_id,
                    "bbox": [bb_left, bb_top, bb_width, bb_height],
                    "area": area,
                    "iscrowd": 0,
                    "segmentation": []
                })
                global_ann_id += 1

    output_file.parent.mkdir(parents=True, exist_ok=True)
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(coco_data, f)

    return len(coco_data["images"]), len(coco_data["annotations"]), skipped_annotations, track_spans

In [ ]:
# Define the output path for your COCO JSON
output_json = data_dir / "annotations" / "instances_default.json"

# Run the refactored conversion function
total_imgs, total_anns, skipped_anns, coco_tracking_data = convert_bambi_to_coco(
    data_dir=data_dir, 
    output_file=output_json,
    visibility_threshold=0.3,
    skip_propagated=True
)

# Print the results
print("--- COCO Conversion Complete ---")
print(f"JSON saved to: {output_json}")
print(f"Total Images Registered: {total_imgs}")
print(f"Total Annotations Saved: {total_anns}")
print(f"Skipped Annotations (Filters): {skipped_anns}")
print(f"Total Unique Animals Tracked: {len(coco_tracking_data)}")